<a href="https://colab.research.google.com/github/engMohamedAbdAlslam/DRP_segmentation/blob/copilot%2Fdevelop-preprocessing-pipeline/notebooks/04_dr_grading_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 04 — DR Grading Model (EfficientNet-B4)
**Dataset:** APTOS 2019 (preprocessed `.npz` from NB01)
**Goal:** Train EfficientNet-B4 to classify DR severity into 5 grades (0–4).
**Metrics:** Accuracy, Quadratic Weighted Kappa (QWK), F1-score, AUC-ROC
> Run on **Google Colab** with T4 GPU. Drive must be mounted with NB01 output at `DRP_processed/aptos2019/`.

## 1. Colab Repo Setup

In [1]:
import os
from pathlib import Path

repo_path = Path('/content/DRP_segmentation')
if not repo_path.exists():
    !git clone https://github.com/engMohamedAbdAlslam/DRP_segmentation.git /content/DRP_segmentation
%cd /content/DRP_segmentation
!git checkout copilot/develop-preprocessing-pipeline
!git pull origin copilot/develop-preprocessing-pipeline
print('Repo ready at', Path.cwd())

Cloning into '/content/DRP_segmentation'...
remote: Enumerating objects: 259, done.
remote: Counting objects: 100% (259/259), done.
remote: Compressing objects: 100% (186/186), done.
remote: Total 259 (delta 131), reused 162 (delta 61), pack-reused 0 (from 0)
Receiving objects: 100% (259/259), 3.80 MiB | 10.38 MiB/s, done.
Resolving deltas: 100% (131/131), done.
/content/DRP_segmentation
Branch 'copilot/develop-preprocessing-pipeline' set up to track remote branch 'copilot/develop-preprocessing-pipeline' from 'origin'.
Switched to a new branch 'copilot/develop-preprocessing-pipeline'
From https://github.com/engMohamedAbdAlslam/DRP_segmentation
 * branch            copilot/develop-preprocessing-pipeline -> FETCH_HEAD
Already up to date.
Repo ready at /content/DRP_segmentation


## 2. Install Dependencies

In [2]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install',
                'timm', 'scikit-learn', 'tqdm', 'matplotlib', 'numpy', 'torch', 'torchvision'],
               check=True)
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

PyTorch: 2.11.0+cu128
CUDA available: True
Device: Tesla T4


## 3. Imports & Config

In [3]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import timm
import matplotlib.pyplot as plt
from sklearn.metrics import (cohen_kappa_score, f1_score,
                             classification_report, confusion_matrix,
                             roc_auc_score)
from pathlib import Path
from tqdm import tqdm
import json, shutil

# ── Config ──
DRIVE_BASE    = Path('/content/drive/MyDrive/DRP_processed/aptos2019')
MODEL_NAME    = 'efficientnet_b4'
NUM_CLASSES   = 5
BATCH_SIZE    = 32
NUM_EPOCHS    = 20
LR            = 1e-4
WEIGHT_DECAY  = 1e-4
PATIENCE      = 5       # early stopping
DEVICE        = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED          = 42

torch.manual_seed(SEED)
np.random.seed(SEED)
print(f'Device: {DEVICE} | Model: {MODEL_NAME} | Epochs: {NUM_EPOCHS}')

Device: cuda | Model: efficientnet_b4 | Epochs: 20


## 4. Mount Drive & Verify Data

In [4]:
from google.colab import drive
drive.mount('/content/drive')

for split in ['train', 'val', 'test']:
    p = DRIVE_BASE / split
    count = len(list(p.rglob('*.npz'))) if p.exists() else 0
    status = '✔' if count > 0 else '❌'
    print(f'{status} {split}: {count} files')

Mounted at /content/drive
✔ train: 2051 files
✔ val: 438 files
✔ test: 440 files


## 5. Dataset Class

In [14]:
# شوف أول 3 ملفات بالـ Drive
for f in list((DRIVE_BASE / 'train').glob('*.npz'))[:3]:
    print(f.stem)

# وشوف أول 3 صور بالـ dataset
for f in list(dl.rglob('*.png'))[:3]:
    print(f.stem)

6e92b1c5ac8e
b598bc9753c2
71e4130bf5c8
17f6c7072f61
0243404e8a00
0083ee8054ee


In [15]:
# NB01 حفظ الـ df بالـ processed_dir؟ نشوف
import subprocess
r = subprocess.run(['find', '/content/DRP_segmentation/data', '-name', '*.csv'],
                   capture_output=True, text=True)
print(r.stdout)

# وكمان بالـ Drive
r2 = subprocess.run(['find', '/content/drive/MyDrive/DRP_processed', '-name', '*.csv'],
                    capture_output=True, text=True)
print(r2.stdout)


/content/drive/MyDrive/DRP_processed/dr_resized_val_index.csv
/content/drive/MyDrive/DRP_processed/dr_resized_test_index.csv



In [17]:
from google.colab import userdata
import os, json

try:
    token_raw = userdata.get('KAGGLE_TOKEN')
    if token_raw and token_raw.strip().startswith('{'):
        t = json.loads(token_raw)
        os.environ['KAGGLE_USERNAME'] = t['username']
        os.environ['KAGGLE_KEY']      = t['key']
    else:
        os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
        os.environ['KAGGLE_KEY']      = userdata.get('KAGGLE_KEY')
except:
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY']      = userdata.get('KAGGLE_KEY')

print(f"User: {os.environ.get('KAGGLE_USERNAME')}")


import os
os.makedirs('/content/aptos_csv', exist_ok=True)
!kaggle datasets download -d mariaherrerot/aptos2019 -f trainLabels.csv -p /content/aptos_csv/ --unzip

import pandas as pd
df = pd.read_csv('/content/aptos_csv/trainLabels.csv')
print(df.columns.tolist())
print(df.head(3))

SecretNotFoundError: Secret KAGGLE_KEY does not exist.

In [11]:
import kagglehub, pandas as pd, shutil
from pathlib import Path
from tqdm import tqdm

dl  = Path(kagglehub.dataset_download('mariaherrerot/aptos2019'))
csv = next(dl.rglob('train.csv'))
df  = pd.read_csv(csv)

# map: id_code → diagnosis
grade_map = dict(zip(df['id_code'].astype(str), df['diagnosis'].astype(int)))
print(f'Grade map loaded: {len(grade_map)} entries')

# Rename each split
renamed, skipped, notfound = 0, 0, 0
for split in ['train', 'val', 'test']:
    split_dir = DRIVE_BASE / split
    for f in tqdm(list(split_dir.glob('*.npz')), desc=split):
        stem = f.stem
        if '_grade' in stem:
            skipped += 1
            continue
        grade = grade_map.get(stem)
        if grade is None:
            notfound += 1
            continue
        new_name = f.parent / f'{stem}_grade{grade}.npz'
        f.rename(new_name)
        renamed += 1

print(f'\nRenamed: {renamed} | Already done: {skipped} | Not found: {notfound}')

Using Colab cache for faster access to the 'aptos2019' dataset.


StopIteration: 

In [9]:
class APTOSDataset(Dataset):
    """Loads preprocessed .npz files from NB01.
    Each file contains: image (H,W,3) float32 imagenet-normalized, metadata JSON.
    Label is extracted from filename: *_grade{N}.npz
    """
    def __init__(self, split_dir: Path, augment: bool = False):
        self.files = sorted(split_dir.rglob('*.npz'))
        self.augment = augment
        self.aug_transform = transforms.Compose([
            transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(),
            transforms.RandomRotation(15),
            transforms.ColorJitter(brightness=0.2, contrast=0.2),
        ]) if augment else None
        assert len(self.files) > 0, f'No .npz files found in {split_dir}'

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        data  = np.load(self.files[idx], allow_pickle=True)
        image = data['image'].astype(np.float32)  # (H, W, 3)
        # Extract grade from filename: stem ends with _grade{N}
        stem  = self.files[idx].stem              # e.g. '0a4e1a29_grade2'
        label = int(stem.split('_grade')[-1])
        # HWC -> CHW for PyTorch
        image = torch.from_numpy(image).permute(2, 0, 1)  # (3, H, W)
        if self.augment and self.aug_transform:
            image = self.aug_transform(image)
        return image, label

# Build datasets
train_ds = APTOSDataset(DRIVE_BASE / 'train', augment=True)
val_ds   = APTOSDataset(DRIVE_BASE / 'val',   augment=False)
test_ds  = APTOSDataset(DRIVE_BASE / 'test',  augment=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}')
# Class distribution
from collections import Counter
labels = [int(f.stem.split('_grade')[-1]) for f in train_ds.files]
print('Train class dist:', dict(sorted(Counter(labels).items())))

Train: 2051 | Val: 438 | Test: 440


ValueError: invalid literal for int() with base 10: '1b329a127307'

In [6]:

for f in train_ds.files[:5]:
    print(f.stem)

1b329a127307
1b32e1d775ea
1b3647865779
1b495ac025b7
1b862fb6f65d


In [7]:
# افحص محتوى ملف واحد
data = np.load(train_ds.files[0], allow_pickle=True)
print('Keys:', list(data.keys()))
import json
meta = json.loads(str(data['metadata']))
print('Metadata:', meta)

Keys: ['image', 'metadata']
Metadata: {'original_shape': [1958, 2588, 3], 'cropped_shape': [1958, 2588, 3], 'crop_bbox': [0, 0, 2588, 1958], 'resized_shape': [512, 512, 3], 'normalization': 'zero_one', 'config': {'target_size': [512, 512], 'crop_border': True, 'illumination_correction': True, 'clahe_clip_limit': 2.0, 'clahe_tile_grid_size': [8, 8], 'normalization': 'zero_one', 'channel_first': False, 'foreground_threshold': 10}}


## 6. Model — EfficientNet-B4

In [ ]:
model = timm.create_model(MODEL_NAME, pretrained=True, num_classes=NUM_CLASSES)
model = model.to(DEVICE)

# Class weights for imbalanced dataset
from sklearn.utils.class_weight import compute_class_weight
class_weights = compute_class_weight('balanced',
                                      classes=np.arange(NUM_CLASSES),
                                      y=labels)
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)
print('Class weights:', class_weights.cpu().numpy().round(3))

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

total_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f'Model: {MODEL_NAME} | Params: {total_params:.1f}M')

## 7. Training Loop

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None, device=DEVICE):
    training = optimizer is not None
    model.train() if training else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []
    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for images, labels_batch in tqdm(loader, leave=False):
            images = images.to(device)
            labels_batch = labels_batch.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels_batch)
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            preds = outputs.argmax(dim=1)
            total_loss += loss.item() * images.size(0)
            correct    += (preds == labels_batch).sum().item()
            total      += images.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels_batch.cpu().numpy())
    avg_loss = total_loss / total
    acc      = correct / total
    qwk      = cohen_kappa_score(all_labels, all_preds, weights='quadratic')
    return avg_loss, acc, qwk, all_preds, all_labels

# ── Training ──
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': [], 'train_qwk': [], 'val_qwk': []}
best_qwk    = -1.0
patience_ct = 0
best_path   = Path('/content/best_efficientnet_b4.pth')

for epoch in range(1, NUM_EPOCHS + 1):
    tr_loss, tr_acc, tr_qwk, _, _ = run_epoch(model, train_loader, criterion, optimizer)
    vl_loss, vl_acc, vl_qwk, _, _ = run_epoch(model, val_loader,   criterion)
    scheduler.step()
    history['train_loss'].append(tr_loss); history['val_loss'].append(vl_loss)
    history['train_acc'].append(tr_acc);   history['val_acc'].append(vl_acc)
    history['train_qwk'].append(tr_qwk);   history['val_qwk'].append(vl_qwk)
    print(f'Epoch {epoch:02d}/{NUM_EPOCHS} | '          f'Loss {tr_loss:.4f}/{vl_loss:.4f} | '          f'Acc {tr_acc:.3f}/{vl_acc:.3f} | '          f'QWK {tr_qwk:.3f}/{vl_qwk:.3f}')
    if vl_qwk > best_qwk:
        best_qwk = vl_qwk
        torch.save(model.state_dict(), best_path)
        patience_ct = 0
        print(f'  ✓ New best QWK: {best_qwk:.4f} — model saved')
    else:
        patience_ct += 1
        if patience_ct >= PATIENCE:
            print(f'Early stopping at epoch {epoch}')
            break

print(f'\nBest Val QWK: {best_qwk:.4f}')

## 8. Training Curves

In [ ]:
epochs_ran = range(1, len(history['train_loss']) + 1)
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(epochs_ran, history['train_loss'], label='Train'); axes[0].plot(epochs_ran, history['val_loss'], label='Val')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].set_xlabel('Epoch')

axes[1].plot(epochs_ran, history['train_acc'], label='Train'); axes[1].plot(epochs_ran, history['val_acc'], label='Val')
axes[1].set_title('Accuracy'); axes[1].legend(); axes[1].set_xlabel('Epoch')

axes[2].plot(epochs_ran, history['train_qwk'], label='Train'); axes[2].plot(epochs_ran, history['val_qwk'], label='Val')
axes[2].set_title('Quadratic Weighted Kappa'); axes[2].legend(); axes[2].set_xlabel('Epoch')

plt.suptitle('EfficientNet-B4 — Training History', fontsize=13)
plt.tight_layout(); plt.show()

## 9. Evaluation on Test Set

In [ ]:
# Load best model
model.load_state_dict(torch.load(best_path, map_location=DEVICE))
_, test_acc, test_qwk, test_preds, test_labels = run_epoch(model, test_loader, criterion)

DR_NAMES = ['No DR', 'Mild', 'Moderate', 'Severe', 'Proliferative']
print(f'Test Accuracy : {test_acc:.4f}')
print(f'Test QWK      : {test_qwk:.4f}')
print(f'Test F1 (macro): {f1_score(test_labels, test_preds, average="macro"):.4f}')
print('\nClassification Report:')
print(classification_report(test_labels, test_preds, target_names=DR_NAMES))

# Confusion Matrix
import seaborn as sns
cm = confusion_matrix(test_labels, test_preds)
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=DR_NAMES, yticklabels=DR_NAMES, ax=ax)
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix — EfficientNet-B4 DR Grading')
plt.tight_layout(); plt.show()

## 10. Save Model to Google Drive

In [ ]:
models_dir = Path('/content/drive/MyDrive/DRP_models')
models_dir.mkdir(parents=True, exist_ok=True)

dest = models_dir / 'efficientnet_b4_dr_grading.pth'
shutil.copy(best_path, dest)

# Save training metadata
meta = {
    'model': MODEL_NAME,
    'num_classes': NUM_CLASSES,
    'best_val_qwk': round(best_qwk, 4),
    'test_acc': round(test_acc, 4),
    'test_qwk': round(test_qwk, 4),
    'epochs_trained': len(history['train_loss']),
    'input_size': '512x512',
    'normalization': 'imagenet',
    'dataset': 'APTOS 2019'
}
with open(models_dir / 'efficientnet_b4_dr_grading_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)

print(f'Model saved → {dest}')
print(f'Metadata: {meta}')

## Next Steps
- **NB07:** Apply Grad-CAM on this model to generate explainability heatmaps
- **NB08:** Load this model in the Gradio clinical interface
- Model path: `DRP_models/efficientnet_b4_dr_grading.pth`